## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Sneaker listings: Masuyama's rejection limits

**Outlier Test (Smirnov-Grubbs)** · a worked example from the StatNex teaching guide

**The story.** A reseller scraped 47 online listings for the same sneaker model to set a fair price. Two listings show no price, and one asks \$1,999.

**The question.** Is the \$1,999 listing an outlier, and is any other price suspicious?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `01_basic/outlier_test.csv` (47 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Listing,Price ($)
Listing 1,183.0
Listing 2,147.0
Listing 3,171.0
Listing 4,142.0
Listing 5,162.0
Listing 6,162.0
Listing 7,174.0
Listing 8,156.0
Listing 9,174.0
Listing 10,152.0
Listing 11,156.0
Listing 12,150.0
Listing 13,
Listing 14,130.0
Listing 15,174.0
Listing 16,141.0
Listing 17,166.0
Listing 18,168.0
Listing 19,180.0
Listing 20,175.0
Listing 21,149.0
Listing 22,171.0
Listing 23,179.0
Listing 24,187.0
Listing 25,158.0
Listing 26,158.0
Listing 27,169.0
Listing 28,148.0
Listing 29,173.0
Listing 30,183.0
Listing 31,1999.0
Listing 32,175.0
Listing 33,176.0
Listing 34,184.0
Listing 35,156.0
Listing 36,146.0
Listing 37,139.0
Listing 38,171.0
Listing 39,181.0
Listing 40,171.0
Listing 41,179.0
Listing 42,
Listing 43,175.0
Listing 44,144.0
Listing 45,154.0
Listing 46,133.0
Listing 47,169.0
""")

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 140)

## The data

In [ ]:
# One column per variable. Blank and text cells are skipped but keep their row
# number, so a result can be traced back to its row.
# Every test below assumes the values other than the candidate come from one normal
# distribution. A small P says the candidate sits further out than such a sample
# would plausibly put its most extreme value; it cannot say why (a typo, a fault, a
# genuine rare case), so check the record before deleting anything, and report the
# results with and without it. Testing one value at a time can miss two outliers
# lying close together, which inflate the SD between them (masking).
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Price ($)"]
alpha = 5 / 100

## Rejection limits (Masuyama)

In [ ]:
# Nothing is tested here. The limits are the range a further value from the same
# normal population should fall in: mean ± t(alpha/2, n - 1) × SD × √(1 + 1/n), the
# prediction interval for one new value. A later value outside it is rejected as not
# belonging to the population.
limits = {}
for name in columns:
    x = pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float)
    n, mean, sd = x.size, np.mean(x), np.std(x, ddof=1)
    half = stats.t.isf(alpha / 2, n - 1) * sd * np.sqrt(1 + 1 / n)
    limits[name] = {"n": n, "mean": mean, "sd": sd,
                    "lower limit": mean - half, "upper limit": mean + half}
limits = pd.DataFrame(limits).T
print(limits)

## What the result says

Masuyama's method does not test one value at a time. It gives two rejection limits, and any value outside them counts as an outlier, a value too extreme to belong with the rest. The limits come from the mean (average) and the standard deviation (typical spread) of all 45 prices: about \$204 and \$274.

The lower limit is -354.16 and the upper limit is 762.61. The \$1,999 listing is far above the upper limit, so it is an outlier. No other price is outside the limits.

Look at how wide the limits are. The \$1,999 listing itself makes the standard deviation very large, so the lower limit is below zero and no low price could ever be rejected.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.